In [1]:
from sentence_transformers import SentenceTransformer

# Carregar modelo multilíngue (suporta português)
modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

frase1 = "Projeto de subestação 138kV para indústria no Pará"
frase2 = "Dimensionamento de subestação de alta tensão industrial"
frase3 = "Receita de bolo de chocolate"

vetores = modelo.encode([frase1, frase2, frase3])

print(f"Shape dos vetores: {vetores.shape}")
print(f"Primeiros 5 números do vetor 1: {vetores[0][:5].round(4)}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Shape dos vetores: (3, 384)
Primeiros 5 números do vetor 1: [ 0.0416  0.0773 -0.1015 -0.2965  0.0345]


In [2]:
from sentence_transformers import util

# Calcular similaridade entre as frases
sim_1_2 = util.cos_sim(vetores[0], vetores[1]).item()
sim_1_3 = util.cos_sim(vetores[0], vetores[2]).item()

print(f"Frase 1: 'Subestação 138kV para indústria no Pará'")
print(f"Frase 2: 'Dimensionamento de subestação de alta tensão'")
print(f"Frase 3: 'Receita de bolo de chocolate'")
print()
print(f"Similaridade frase1 x frase2: {sim_1_2:.4f}  ← deveria ser ALTA")
print(f"Similaridade frase1 x frase3: {sim_1_3:.4f}  ← deveria ser BAIXA")

Frase 1: 'Subestação 138kV para indústria no Pará'
Frase 2: 'Dimensionamento de subestação de alta tensão'
Frase 3: 'Receita de bolo de chocolate'

Similaridade frase1 x frase2: 0.4611  ← deveria ser ALTA
Similaridade frase1 x frase3: -0.0742  ← deveria ser BAIXA


In [3]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer, util

# Base de projetos PCM Modesto
projetos = [
    "Subestação compacta 138kV para indústria cerâmica em Marabá PA",
    "SPDA categoria I para galpão industrial em Belém PA",
    "Projeto fotovoltaico 150kWp para supermercado em Ananindeua PA",
    "Subestação 13,8kV com transformador 500kVA para escola pública",
    "Instalação elétrica predial apartamentos residenciais Belém PA",
    "Projeto elétrico BIM para auditório 415 lugares Oriximiná PA",
    "Dimensionamento de HVAC 65TR para auditório público PA",
    "Rede de distribuição rural MT/BT Luz Para Todos Pará",
]

# Gerar vetores dos projetos
vetores_projetos = modelo.encode(projetos).astype('float32')

# Criar índice FAISS
indice = faiss.IndexFlatL2(vetores_projetos.shape[1])
indice.add(vetores_projetos)
print(f"✅ Projetos indexados: {indice.ntotal}")

# Fazer consulta
consulta = "subestação alta tensão indústria Pará"
vetor_consulta = modelo.encode([consulta]).astype('float32')

# Buscar top 3
distancias, indices = indice.search(vetor_consulta, 3)

print(f"\n🔍 Consulta: '{consulta}'")
print("\n📋 Top 3 projetos mais similares:\n")
for i in range(3):
    print(f"  {i+1}. {projetos[indices[0][i]]}")
    print(f"     Distância: {distancias[0][i]:.4f}\n")

✅ Projetos indexados: 8

🔍 Consulta: 'subestação alta tensão indústria Pará'

📋 Top 3 projetos mais similares:

  1. SPDA categoria I para galpão industrial em Belém PA
     Distância: 13.3882

  2. Subestação compacta 138kV para indústria cerâmica em Marabá PA
     Distância: 17.8131

  3. Rede de distribuição rural MT/BT Luz Para Todos Pará
     Distância: 21.0387



In [5]:
# Salvar índice
faiss.write_index(indice, "indice_projetos_pcm.faiss")
print("Índice salvo!")

# Carregar depois
indice_carregado = faiss.read_index("indice_projetos_pcm.faiss")
print(f"Índice carregado com {indice_carregado.ntotal} projetos")


Índice salvo!
Índice carregado com 8 projetos
